# Objetivo

Desenvolvimento do Teleutils em ambiente de teste local

# Configuração do Ambiente

## Bibliotecas

In [ ]:
# 1. Módulos da Biblioteca Padrão
import logging
from pathlib import Path

# 2. Módulos de Terceiros
import pandas as pd

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.functions import pandas_udf
from pyspark.sql import types as T # ArrayType, StructType, StructField, StringType, BooleanType

# 3. Teletools/Teleutils
from teleutils.core.extractors import CDRParquetExtractor, CDRTextExtractor
from teleutils.core.transformers import CDRTransformer

In [ ]:
# Configuração do Pandas
# Exibir todas as colunas
pd.set_option("display.max_columns", None)

# Exibir todas as linhas
pd.set_option("display.max_rows", None)

# Evitar que o texto nas células seja cortado
pd.set_option("display.max_colwidth", None)

In [ ]:
# Configuração para exibir logs no output da célula
logging.basicConfig(
 level=logging.INFO,
 format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
 datefmt="%H:%M:%S",
)

# Reduzir ruído de bibliotecas externas (pyspark, py4j, etc.)
logging.getLogger("py4j").setLevel(logging.WARNING)
logging.getLogger("pyspark").setLevel(logging.WARNING)

## Spark

In [ ]:
spark = (
 SparkSession
 .builder
 .appName("SessaoLocal")
 .master("local[12]")
 .config("spark.driver.memory", "24g")
 .getOrCreate()
)
spark

## Arquivos de dados

### Dados de origem

In [ ]:
folder_claro = Path("/assets/monitoring/cdr/cdr_bruto/Semana117/Claro/home/mnt_srt2e/SRTEBT/coleta/coletados/")
_files_claro_lega = [(str(file), str(file.parent.name)) for file in folder_claro.rglob("*TRI*/*LEGA*.gz") if file.is_file()]
df_lega = pd.DataFrame(_files_claro_lega, columns=["file_path", "parent_folder"])
df_lega["exchange_id"] = df_lega["parent_folder"].str.slice(stop=4)
files_claro_lega = df_lega.groupby("exchange_id").sample(15,random_state=42)["file_path"].to_list()
files_claro_lega[:5]

In [ ]:
folder_claro = Path("/assets/monitoring/cdr/cdr_bruto/Semana117/Claro/home/mnt_srt2e/SRTEBT/coleta/coletados/")
_files_claro_legb = [(str(file), str(file.parent.name)) for file in folder_claro.rglob("*TRI*/*LEGB*.gz") if file.is_file()]
df_legb = pd.DataFrame(_files_claro_legb, columns=["file_path", "parent_folder"])
df_legb["exchange_id"] = df_legb["parent_folder"].str.slice(stop=4)
files_claro_legb = df_legb.groupby("exchange_id").sample(15,random_state=42)["file_path"].to_list()
files_claro_legb[:5]

### Arquivos intermediário e final

In [ ]:
extracted_claro_lega = "/assets/data/cdr/teletools/db/extraidos/tropico_claro_extraido_lega.parquet"
extracted_claro_legb = "/assets/data/cdr/teletools/db/extraidos/tropico_claro_extraido_legb.parquet"
extracted_claro = "/assets/data/cdr/teletools/db/extraidos/tropico_claro_extraido.parquet"
transformed_claro = "/assets/data/cdr/teletools/db/transformados/tropico_claro_transformado.parquet"

# Análise

## Dados brutos

### LEG A

In [ ]:
df_raw_lega = spark.read.csv(files_claro_lega, sep=",")
df_raw_lega.limit(5).toPandas()

In [ ]:
raw_column_names_lega = ("CSN", "Switch ID", "Bill Type", "Start Date", "Start Time", "End Date", "End Time", "Duration", "Calling Number", "Called Number", "Dialed Number", "Billed Party", "Connected Number", "Reserva", "Partial Record Ind.", "Free Ind.", "Charged Party Ind.", "Itrunk", "Otrunk", "Call Type", "GSVN", "Failure Code", "Termination Code", "Bearer Service", "Tele Service", "SIP Status code", "Time Zone", "Caller Address", "Address 1", "Address 2", "Address 3", "Coding Type", "FDS", "Caller Processor", "Reserva", "Clock Indicator", "Partial Counter", "Seizure Date", "Seizure Time", "Release Date", "Release Time", "Caller category", "IN CALLID", "In bill count")
raw_column_indices_lega = list(range(len(raw_column_names_lega)))
raw_columns_to_keep_lega = [
    F.col(df_raw_lega.columns[index]).alias(column_name)
    for index, column_name in zip(
        raw_column_indices_lega,
        raw_column_names_lega,
    )
]
df_raw_lega.select(raw_columns_to_keep_lega).limit(5).toPandas()

In [ ]:
column_indices_lega = (1, 2, 3, 4, 5, 6, 7, 8, 9, 17, 18, 19, 25, 42)
column_names_lega = (
    "bilhetador",
    "resultado_chamada",
    "_data",
    "_hora",
    "_data_fim",
    "_hora_fim",
    "duracao",
    "numero_origem",
    "numero_destino",
    "rota_entrada",
    "rota_saida",
    "tipo_chamada",
    "codigo_resposta_sip",
    "referencia",
)

columns_to_keep_lega = [
    F.col(df_raw_lega.columns[index]).alias(column_name)
    for index, column_name in zip(
        column_indices_lega,
        column_names_lega,
    )
]
df_raw_lega.select(columns_to_keep_lega).limit(5).toPandas()

### LEG B

In [ ]:
df_raw_legb = spark.read.csv(files_claro_legb, sep=",")
df_raw_legb.limit(5).toPandas()

In [ ]:
raw_column_names_legb = ("CSN", "Switch ID", "Bill Type", "ans", "Start Time", "End Date", "End Time", "Duration", "Calling Number", "Dialed Number", "Billed Party", "Partial Record Ind.", "Free Ind.", "Charged Party Ind.", "Itrunk", "Otrunk", "Failure Code", "Caled Address", "Address 1", "Address 2", "Address 3", "Caller category", "Destination_Number", "NOA", "NPI", "SIP status code", "Reserva", "Reserva", "Reserva", "Reserva", "Service Key", "Reserva", "Reserva", "Reserva", "Reserva", "Reserva", "Reserva", "Reserva", "Reserva", "Reserva", "Reserva", "Reserva", "Reserva", "Reserva", "Reserva", "IN CALLID", "Partial Record indicator")
raw_column_indices_legb = list(range(len(raw_column_names_legb)))
raw_columns_to_keep_legb = [
    F.col(df_raw_legb.columns[index]).alias(column_name)
    for index, column_name in zip(
        raw_column_indices_legb,
        raw_column_names_legb,
    )
]
df_raw_legb.select(raw_columns_to_keep_legb).limit(5).toPandas()

In [ ]:
column_indices_legb = (1,2,3,4,5,6,7,8,9,14,15,25,45)
column_names_legb = (
    "bilhetador", 
    "resultado_chamada", 
    "_data", 
    "_hora", 
    "_data_fim", 
    "_hora_fim", 
    "duracao", 
    "numero_origem",
    "numero_destino",
    "rota_entrada",
    "rota_saida",
    "codigo_resposta_sip",
    "referencia",)

columns_to_keep_legb = [
    F.col(df_raw_legb.columns[index]).alias(column_name)
    for index, column_name in zip(
        column_indices_legb,
        column_names_legb,
    )
]
df_raw_legb.select(columns_to_keep_legb).limit(5).toPandas()

### LEG A+B

In [ ]:
display(df_raw_lega.select(columns_to_keep_lega).limit(5).toPandas())
display(df_raw_legb.select(columns_to_keep_legb).limit(5).toPandas())

In [ ]:
df = df_raw_lega.select(columns_to_keep_lega).unionByName(
    df_raw_legb.select(columns_to_keep_legb), 
    allowMissingColumns=True)
df.show(5)

In [ ]:
df_raw_lega.count(),df_raw_legb.count(),

In [ ]:
df.filter(F.col("tipo_chamada").isNull()).count()

In [ ]:
df.groupBy(F.col("tipo_chamada")).count().show()

## Extração

In [ ]:
extractor = CDRTextExtractor(spark)
extractor.extract(files_claro_lega, extracted_claro_lega, "stfc_tropico_claro_lega", operator="claro", cdr_type="tropico_lega")
extractor.extract(files_claro_legb, extracted_claro_legb, "stfc_tropico_claro_legb", operator="claro", cdr_type="tropico_legb")

In [ ]:
df_extracted = spark.read.parquet(extracted_claro_lega).unionByName(spark.read.parquet(extracted_claro_legb), allowMissingColumns=True)
df_extracted.write.mode("overwrite").parquet(extracted_claro)

In [ ]:
df_extracted = spark.read.parquet(extracted_claro)
df_extracted.show(5, truncate=False)

In [ ]:
df_extracted.limit(5).toPandas()

In [ ]:
df_extracted.groupBy(F.col("_tipo_chamada")).count().show()

In [ ]:
prestadora = "claro"
if prestadora:
    operator_expression = F.lit(prestadora)
else:
    operator_expression = F.element_at(F.split(F.input_file_name(), "/"), -3)
    
df_extracted.withColumn("_ttt", operator_expression).select("prestadora", "_ttt").show()

In [ ]:
# # "3", "b-AnswerHasBeenReceived"
# df_extracted.groupBy("_resultado_chamada").count().orderBy("_resultado_chamada").show()

In [ ]:
# df_extracted.printSchema()

In [ ]:
# df_extracted.groupBy("duracao").count().orderBy("duracao").show(5)

## Transformação

In [ ]:
transformer = CDRTransformer(spark)
transformer.transform_stfc_tropico_claro(extracted_claro, transformed_claro)

In [ ]:
df_transformed = spark.read.parquet(transformed_claro)
df_transformed.limit(5).toPandas().dropna(how="all",axis=1)

In [ ]:
# df_transformed.groupBy("no_tipo_chamada").count().show()

In [ ]:
# df_transformed.printSchema()